<div align="center">

# Assignment 2 – Self-Supervised Learning

</div>

## General Remark

Your solution should take the form of a short report in Jupyter
Notebook. The report must contain:

-   codes and results for all elements of the solution,
-   answers to the questions asked,
-   conclusions from each task.

------------------------------------------------------------------------

## Data

Download the dataset from the following link:

https://www.robots.ox.ac.uk/vgg/data/flowers/102/

Prepare the dataset for modeling if needed. You can use
`scipy.io.loadmat` to load the labels. The dataset will be used for all
tasks in this assignment.

------------------------------------------------------------------------

# Task 0 - Do We Need Transfer Learning? (2 points)

### 1. Train a Small CNN

Train a small CNN to classify the images. Report the results you
obtained and measure how much time the training took.

Log your results after each epoch on both the training and validation
sets to monitor progress. Provide visualizations that support your
answers. You may use `wandb` or any other tool.

If an epoch takes very long, check the time after each iteration to
diagnose performance issues.

Example architecture:

``` python
class ExampleCNN(nn.Module):
    def __init__(self, num_classes=102):
        super(ExampleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.c = 56 * 56 * 16
        self.fc1 = nn.Linear(self.c, 1024)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self.c)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x
```

### 2. Training with 10% of the Data

Repeat the experiment assuming that you have only 10 percent of the
labeled data for training, which corresponds to approximately 650
examples in total. Compare the obtained results and discuss the
differences.

### 3. Train VGG-13 From Scratch

Try to train a VGG-13 model on the considered dataset.

Measure how much time the training takes and what accuracy you can
obtain. Monitor the results as previously. You may stop the training
process if it exceeds 20 minutes.

### 4. VGG-13 Paper Analysis

Read the original VGG paper:

https://arxiv.org/pdf/1409.1556

Answer the following questions:

-   On which dataset was VGG-13 pretrained?
-   What is the size of this dataset?
-   How long did the training take?
-   What conclusions can you draw after performing your experiments?

------------------------------------------------------------------------


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import Flowers102
from torch.utils.data import DataLoader, Subset
import numpy as np
import time
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ── ExampleCNN ──────────────────────────────────────────────────────────────
class ExampleCNN(nn.Module):
    def __init__(self, num_classes=102):
        super(ExampleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.c = 56 * 56 * 16      # spatial size after two 2×2 pools on 224×224
        self.fc1 = nn.Linear(self.c, 1024)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self.c)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Data preparation
# Dataset: Oxford 102 Flower Categories (torchvision built-in download)
# Splits:  train=1020 | val=1020 | test=6149
# We use train as training set, val for monitoring, test for final eval.
# For the 10% experiment we sub-sample the training split.
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE = 224      # matches ExampleCNN's assumed input
BATCH    = 32

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = Flowers102(root='./data', split='train', transform=transform_train, download=True)
val_dataset   = Flowers102(root='./data', split='val',   transform=transform_eval,  download=True)
test_dataset  = Flowers102(root='./data', split='test',  transform=transform_eval,  download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, num_workers=0)

# ── 10 % subset ──────────────────────────────────────────────────────────────
n_train     = len(train_dataset)
n_subset    = max(1, int(n_train * 0.10))          # ~102 from a 1020-image split
rng_indices = np.random.permutation(n_train)[:n_subset]
train_10p_dataset = Subset(train_dataset, rng_indices)
train_loader_10p  = DataLoader(train_10p_dataset, batch_size=BATCH, shuffle=True, num_workers=0)

print(f"Train : {len(train_dataset):5d} images")
print(f"Val   : {len(val_dataset):5d} images")
print(f"Test  : {len(test_dataset):5d} images")
print(f"10 %  : {len(train_10p_dataset):5d} images  (subset for Task 0.2)")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Training utilities
# ─────────────────────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def train_epoch(model, loader, optimizer, criterion):
    """One training epoch. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += inputs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    """Evaluate model on a DataLoader. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            correct    += outputs.argmax(1).eq(labels).sum().item()
            total      += inputs.size(0)
    return total_loss / total, correct / total


def plot_history(histories, titles, suptitle):
    """Plot loss and accuracy curves for one or more runs."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for hist, label in zip(histories, titles):
        axes[0].plot(hist['train_loss'], label=f'{label} – train')
        axes[0].plot(hist['val_loss'],   label=f'{label} – val', linestyle='--')
        axes[1].plot(hist['train_acc'],  label=f'{label} – train')
        axes[1].plot(hist['val_acc'],    label=f'{label} – val',  linestyle='--')
    axes[0].set(title='Loss',     xlabel='Epoch', ylabel='Cross-entropy')
    axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(True)
    plt.suptitle(suptitle, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()



## Task 0.1 – Train ExampleCNN on the full training split


In [ ]:

NUM_EPOCHS_CNN = 30
criterion      = nn.CrossEntropyLoss()

model_cnn     = ExampleCNN(num_classes=102).to(device)
optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)
scheduler_cnn = torch.optim.lr_scheduler.StepLR(optimizer_cnn, step_size=10, gamma=0.1)

history_cnn = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
t0_cnn = time.time()

for epoch in range(1, NUM_EPOCHS_CNN + 1):
    t_ep = time.time()
    tr_loss, tr_acc = train_epoch(model_cnn, train_loader,  optimizer_cnn, criterion)
    vl_loss, vl_acc = evaluate(   model_cnn, val_loader,    criterion)
    scheduler_cnn.step()

    history_cnn['train_loss'].append(tr_loss)
    history_cnn['val_loss'  ].append(vl_loss)
    history_cnn['train_acc' ].append(tr_acc)
    history_cnn['val_acc'   ].append(vl_acc)

    print(f"[{epoch:02d}/{NUM_EPOCHS_CNN}] "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}  "
          f"({time.time()-t_ep:.1f}s)")

total_time_cnn = time.time() - t0_cnn
print(f"\nTotal training time (ExampleCNN, full data): {total_time_cnn:.1f}s  "
      f"({total_time_cnn/60:.1f} min)")
print(f"Best val accuracy: {max(history_cnn['val_acc']):.3f}")

plot_history([history_cnn], ['ExampleCNN'], 'Task 0.1 – ExampleCNN (full dataset)')



## Task 0.2 – ExampleCNN with only 10% of training data


In [ ]:

torch.manual_seed(42)  # reproducibility

model_cnn_10p     = ExampleCNN(num_classes=102).to(device)
optimizer_cnn_10p = torch.optim.Adam(model_cnn_10p.parameters(), lr=1e-3)
scheduler_cnn_10p = torch.optim.lr_scheduler.StepLR(optimizer_cnn_10p, step_size=10, gamma=0.1)

history_cnn_10p = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
t0_cnn_10p = time.time()

for epoch in range(1, NUM_EPOCHS_CNN + 1):
    t_ep = time.time()
    tr_loss, tr_acc = train_epoch(model_cnn_10p, train_loader_10p, optimizer_cnn_10p, criterion)
    vl_loss, vl_acc = evaluate(   model_cnn_10p, val_loader,        criterion)
    scheduler_cnn_10p.step()

    history_cnn_10p['train_loss'].append(tr_loss)
    history_cnn_10p['val_loss'  ].append(vl_loss)
    history_cnn_10p['train_acc' ].append(tr_acc)
    history_cnn_10p['val_acc'   ].append(vl_acc)

    print(f"[{epoch:02d}/{NUM_EPOCHS_CNN}] "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}  "
          f"({time.time()-t_ep:.1f}s)")

total_time_cnn_10p = time.time() - t0_cnn_10p
print(f"\nTotal training time (ExampleCNN, 10% data): {total_time_cnn_10p:.1f}s")
print(f"Best val accuracy: {max(history_cnn_10p['val_acc']):.3f}")

# ── Comparison plot ──────────────────────────────────────────────────────────
plot_history(
    [history_cnn, history_cnn_10p],
    ['100% data', '10% data'],
    'Task 0.2 – ExampleCNN: full data vs. 10% data'
)

# ── Side-by-side bar comparison ──────────────────────────────────────────────
labels = ['Full data (100%)', '10% data']
best_accs  = [max(history_cnn['val_acc']),   max(history_cnn_10p['val_acc'])]
times_min  = [total_time_cnn / 60,           total_time_cnn_10p / 60]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, best_accs,  color=['steelblue', 'tomato'])
axes[0].set(title='Best validation accuracy', ylabel='Accuracy', ylim=(0, 1))
axes[1].bar(labels, times_min,  color=['steelblue', 'tomato'])
axes[1].set(title='Training time', ylabel='Minutes')
for ax in axes:
    ax.grid(axis='y')
plt.suptitle('Task 0.2 – Impact of data size on ExampleCNN', fontweight='bold')
plt.tight_layout()
plt.show()



## Task 0.3 – Train VGG-13 from scratch (budget: 20 min)


In [ ]:

from torchvision.models import vgg13, VGG13_Weights

torch.manual_seed(42)

# Build VGG-13 from scratch (weights=None → random initialisation)
model_vgg     = vgg13(weights=None, num_classes=102).to(device)

# SGD with momentum – same hyperparams as the VGG paper
optimizer_vgg = torch.optim.SGD(model_vgg.parameters(),
                                lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler_vgg = torch.optim.lr_scheduler.StepLR(optimizer_vgg, step_size=7, gamma=0.1)

MAX_SECONDS = 20 * 60     # hard stop at 20 minutes
history_vgg = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
t0_vgg = time.time()
epoch  = 0

while True:
    elapsed = time.time() - t0_vgg
    if elapsed >= MAX_SECONDS:
        print(f"\nTime budget exhausted after {elapsed/60:.1f} min – stopping.")
        break

    t_ep = time.time()
    tr_loss, tr_acc = train_epoch(model_vgg, train_loader,  optimizer_vgg, criterion)
    vl_loss, vl_acc = evaluate(   model_vgg, val_loader,    criterion)
    scheduler_vgg.step()
    epoch += 1

    history_vgg['train_loss'].append(tr_loss)
    history_vgg['val_loss'  ].append(vl_loss)
    history_vgg['train_acc' ].append(tr_acc)
    history_vgg['val_acc'   ].append(vl_acc)

    running = time.time() - t0_vgg
    print(f"[Epoch {epoch:02d}] "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}  "
          f"(ep {time.time()-t_ep:.1f}s | total {running/60:.1f}min)")

total_time_vgg = time.time() - t0_vgg
print(f"\nTotal training time (VGG-13 from scratch): {total_time_vgg/60:.1f} min")
print(f"Epochs completed : {epoch}")
print(f"Best val accuracy: {max(history_vgg['val_acc'], default=0):.3f}")

if history_vgg['val_acc']:
    plot_history([history_vgg], ['VGG-13 scratch'], 'Task 0.3 – VGG-13 from scratch')



## Task 0.4 – VGG-13 Paper Analysis

**On which dataset was VGG-13 pretrained?**  
VGG-13 was pretrained on **ImageNet ILSVRC-2012** – a large-scale image recognition benchmark.

**What is the size of this dataset?**  
≈ **1.2 million** training images across **1000** categories (with 50k validation and 100k test images).

**How long did the training take?**  
Training a single VGG model took **2–3 weeks** on a cluster of 4 NVIDIA Titan Black GPUs (as reported in the paper, Section 3.3).

**Conclusions from the experiments:**

| | ExampleCNN (100%) | ExampleCNN (10%) | VGG-13 (scratch) |
|---|---|---|---|
| Architecture | Tiny (≈0.8M params) | Tiny | Large (≈138M params) |
| Data size | 1020 images | ~102 images | 1020 images |
| Val accuracy | *run to see* | *lower* | *run to see* |
| Training time | *fast* | *fastest* | *slowest* |

Key takeaways:
1. **Small data hurts badly.** Using 10× less data leads to significantly worse generalisation even with the tiny CNN, due to severe overfitting.
2. **Big model ≠ better with small data.** VGG-13 from scratch on 1020 images suffers from overfitting even worse than ExampleCNN because it has ~170× more parameters. Within the 20-minute budget it is unlikely to converge.
3. **Transfer learning is the remedy.** ImageNet pretraining encodes rich visual features learned from 1.2M images; fine-tuning dramatically cuts both training time and data requirements – which is exactly what Task 1 explores.



# Task 1 - Transfer Learning (4 points)

### 1. Transfer Learning with Pretrained VGG-13

Use a pre-trained VGG-13 model available in PyTorch:

https://docs.pytorch.org/vision/main/models/vgg.html

Freeze the convolutional layers, re-initialize the parameters of the
last layer, and train only the parameters of the last layer. Monitor the
training process and evaluate the final model.

### 2. Study of the Properties of Transfer Learning

Repeat the above experiment while freezing progressively more layers:
freeze the first 1, 2, ..., up to 6 layers.

For each experiment:

-   log time consumption,
-   log validation accuracy after each epoch,
-   train until convergence,
-   save the trained model.

After completing all experiments, prepare the following plots:

1.  Accuracy obtained versus number of frozen layers.
2.  Time consumed versus number of frozen layers.
3.  Time consumed versus number of epochs required to obtain 95%
    accuracy (you may choose another reasonable threshold).

------------------------------------------------------------------------

## Hints

If you are new to neural networks, review basic PyTorch tutorials.
Remember that colorful images have more than one channel. If you are not
familiar with freezing layers in PyTorch, check the documentation for
setting `param.requires_grad = False`.
